In [ ]:
import numpy as np
import scipy.stats as si
from scipy.stats import norm
from scipy.optimize import brentq

def black_scholes_call(S, K, T, r, sigma):
    #Calculate the price of a European call option using the Black-Scholes formula.
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

def implied_volatility(S, K, T, r, C):
    #Calculate the implied volatility of an option using the Black-Scholes formula.
    def f(sigma):
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2) - C
    try:
        return brentq(f, 1e-6, 1)
    except ValueError:
        return np.nan
    
def calculate_vix(option_data, spot_price, risk_free_rate, time_to_expiry):
    #Calculate the VIX index for a given set of options data for one stock in the S&P500.
    at_the_money_strikes = []
    at_the_money_mid_prices = []
    for option in option_data:
        if abs(option['strike'] - spot_price) < 1e-6:
            at_the_money_strikes.append(option['strike'])
            at_the_money_mid_prices.append(option['mid_price'])
    
    # Check for division by zero
    if len(at_the_money_mid_prices) == 0:
        weighted_implied_vols = 0
    else:
        total_weight = sum(1 / price**2 for price in at_the_money_mid_prices)
        weights = [(1 / price**2) / total_weight for price in at_the_money_mid_prices]
        implied_vols = [implied_volatility(spot_price, strike, time_to_expiry, risk_free_rate, mid_price) 
                       for strike, mid_price in zip(at_the_money_strikes, at_the_money_mid_prices)]
        weighted_implied_vols = sum(weight * iv for weight, iv in zip(weights, implied_vols))

    return 100 * np.sqrt(365 / time_to_expiry) * weighted_implied_vols

# Example usage:
option_data = [{'strike': 200, 'mid_price': 4.20},
               {'strike': 210, 'mid_price': 3.10},
               {'strike': 220, 'mid_price': 2.10},
               {'strike': 230, 'mid_price': 1.40},
               {'strike': 240, 'mid_price': 0.90}]
spot_price = 220
risk_free_rate = 0.01
time_to_expiry = 30 / 365

vix = calculate_vix(option_data, spot_price, risk_free_rate, time_to_expiry)
print(vix)

In [ ]:
import numpy as np
import scipy.stats as si
from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime

def black_scholes_call(S, K, T, r, sigma):
    #Calculate the price of a European call option using the Black-Scholes formula.
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

def implied_volatility(S, K, T, r, C):
    #Calculate the implied volatility of an option using the Black-Scholes formula.
    def f(sigma):
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2) - C
    try:
        return brentq(f, 1e-6, 1)
    except ValueError:
        return np.nan
    
def calculate_vix(option_data, spot_price, risk_free_rate, maturity_date):
    #Calculate the VIX index for a given set of options data for one stock in the S&P500.
    at_the_money_strikes = []
    at_the_money_mid_prices = []
    for option in option_data:
        if option['maturity'] == maturity_date and abs(option['strike'] - spot_price) < 1e-6:
            at_the_money_strikes.append(option['strike'])
            at_the_money_mid_prices.append(option['mid_price'])
    
    # Check for division by zero
    if len(at_the_money_mid_prices) == 0:
        weighted_implied_vols = 0
    else:
        time_to_expiry = (datetime.strptime(maturity_date, '%Y-%m-%d') - datetime.now()).days / 365.0
        total_weight = sum(1 / price**2 for price in at_the_money_mid_prices)
        weights = [(1 / price**2) / total_weight for price in at_the_money_mid_prices]
        implied_vols = [implied_volatility(spot_price, strike, time_to_expiry, risk_free_rate, mid_price) 
                       for strike, mid_price in zip(at_the_money_strikes, at_the_money_mid_prices)]
        weighted_implied_vols = sum(weight * iv for weight, iv in zip(weights, implied_vols))

    return 100 * np.sqrt(365 / time_to_expiry) * weighted_implied_vols

# Example usage:
option_data = [{'strike': 200, 'mid_price': 4.20, 'maturity': '2023-06-01'},
               {'strike': 210, 'mid_price': 3.10, 'maturity': '2023-06-01'},
               {'strike': 220, 'mid_price': 2.10, 'maturity': '2023-06-01'}]
              


In [ ]:
calculate_vix(option_data, spot_price, risk_free_rate,'2023-06-01')